# Run COMPASS GAM stages (R environment)

This notebook isolates all `mgcv`/R work from the Python run notebooks. Select an R conda kernel containing `mgcv` and `data.table`.

Required order:

1. Run the Python local notebook once to build prediction inputs.
2. Run **Stage A** here to create trajectory features.
3. Return to the Python notebook, set `REBUILD_PREDICTION_INPUTS = False`, and rerun its Stage 3 cell. This refits Python models with the GAM features without deleting them.
4. Return here and run **Stage B** for Cox nonlinearity tests.

Stage B rejects feature-selection files older than the trajectory features, guarding against accidentally testing the pre-GAM Python feature list.


In [ ]:
PROJECT_ROOT <- "/data/gusev/USERS/jpconnor/code/CAIA"
SURVIVAL_DIR <- file.path(PROJECT_ROOT, "COMPASS", "survival_analysis")

# Choose "profile_data" for the merged Parquets or "baseline" for ALL_2025_03.
DATA_VARIANT <- "profile_data"
ARM <- "adt"
LANDMARK_DAYS <- c(0L, 90L, 180L)
FORCE_RERUN <- TRUE

DATA_ROOT <- switch(
  DATA_VARIANT,
  profile_data = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS_PROFILE_DATA",
  baseline = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS",
  stop(sprintf("Unknown DATA_VARIANT: %s", DATA_VARIANT))
)
INPUTS_DIR <- file.path(DATA_ROOT, "survival_analysis", sprintf("prediction_inputs_%s", ARM))
MODEL_OUTPUT_DIR <- file.path(DATA_ROOT, "survival_analysis", sprintf("local_runs_%s", ARM))
NONLINEAR_OUTPUT_DIR <- file.path(MODEL_OUTPUT_DIR, "cox", "gam_nonlinearity")
RSCRIPT <- file.path(R.home("bin"), "Rscript")

run_r_script <- function(script_name, args) {
  script_path <- file.path(SURVIVAL_DIR, script_name)
  if (!file.exists(script_path)) stop(sprintf("Missing R script: %s", script_path))
  command_args <- shQuote(c(script_path, args))
  cat(sprintf("[run ] %s %s\n", RSCRIPT, paste(command_args, collapse = " ")))
  status <- system2(RSCRIPT, args = command_args, stdout = "", stderr = "")
  if (!identical(status, 0L)) stop(sprintf("%s failed with status %s", script_name, status))
  invisible(status)
}

cat(sprintf("R:                 %s\n", R.version.string))
cat(sprintf("Rscript:           %s\n", RSCRIPT))
cat(sprintf("data variant:      %s\n", DATA_VARIANT))
cat(sprintf("prediction inputs: %s\n", INPUTS_DIR))
cat(sprintf("model outputs:     %s\n", MODEL_OUTPUT_DIR))


## Stage A — hierarchical trajectory GAM features

Run this after `build_prediction_inputs` has created the pre-treatment long tables and canonical-lab file.


In [ ]:
required_inputs <- c(
  file.path(INPUTS_DIR, "canonical_labs_train_val.csv"),
  file.path(INPUTS_DIR, sprintf("pre_treatment_lab_long_landmark%d.csv", LANDMARK_DAYS))
)
missing_inputs <- required_inputs[!file.exists(required_inputs)]
if (length(missing_inputs)) {
  stop(sprintf("Missing Python-built input(s):\n%s", paste(missing_inputs, collapse = "\n")))
}

trajectory_outputs <- file.path(
  INPUTS_DIR, sprintf("gam_trajectory_features_landmark%d.csv", LANDMARK_DAYS)
)
diagnostic_outputs <- file.path(
  INPUTS_DIR, sprintf("gam_fit_diagnostics_landmark%d.csv", LANDMARK_DAYS)
)
if (FORCE_RERUN || !all(file.exists(c(trajectory_outputs, diagnostic_outputs)))) {
  run_r_script(
    "gam_trajectory_features.R",
    c(
      "--inputs-dir", INPUTS_DIR,
      "--landmark-days", paste(LANDMARK_DAYS, collapse = ","),
      "--k-pop", "10",
      "--k-pat", "5",
      "--trailing-window-days", "180",
      "--nthreads", "1",
      "--fit-split", "all"
    )
  )
} else {
  cat("[skip] all trajectory feature and diagnostic files already exist\n")
}
stopifnot(all(file.exists(c(trajectory_outputs, diagnostic_outputs))))
file.info(trajectory_outputs)[, c("size", "mtime"), drop = FALSE]


## Python handoff — stop here

Return to the matching Python run notebook. Set `REBUILD_PREDICTION_INPUTS = False`, keep `FORCE_RERUN = True`, and rerun Stage 3. The Python univariate and multivariable models will then merge the trajectory feature CSVs automatically. Do not rerun `build_prediction_inputs`, because its cleanup deliberately removes old GAM feature files.


## Stage B — nonlinear Cox GAM tests

Run only after completing the Python handoff above. Each landmark uses the exact feature-selection CSV from its ordinary (`both`) univariate run.


In [ ]:
dir.create(NONLINEAR_OUTPUT_DIR, recursive = TRUE, showWarnings = FALSE)
nonlinear_outputs <- character(length(LANDMARK_DAYS))

for (i in seq_along(LANDMARK_DAYS)) {
  landmark <- LANDMARK_DAYS[[i]]
  trajectory_path <- file.path(INPUTS_DIR, sprintf("gam_trajectory_features_landmark%d.csv", landmark))
  selection_path <- file.path(
    MODEL_OUTPUT_DIR, "cox", sprintf("landmark_%d", landmark), "both",
    "cox_agg_feature_selection.csv"
  )
  if (!file.exists(trajectory_path)) stop(sprintf("Missing %s; run Stage A first.", trajectory_path))
  if (!file.exists(selection_path)) stop(sprintf("Missing %s; rerun Python Stage 3 first.", selection_path))

  trajectory_mtime <- file.info(trajectory_path)$mtime
  selection_mtime <- file.info(selection_path)$mtime
  if (selection_mtime < trajectory_mtime) {
    stop(sprintf(
      paste0(
        "Landmark %d feature selection predates its GAM trajectory features. ",
        "Set REBUILD_PREDICTION_INPUTS = False and FORCE_RERUN = True, then rerun Python Stage 3."
      ),
      landmark
    ))
  }

  output_path <- file.path(
    NONLINEAR_OUTPUT_DIR, sprintf("gam_cox_nonlinearity_landmark%d.csv", landmark)
  )
  nonlinear_outputs[[i]] <- output_path
  if (FORCE_RERUN || !file.exists(output_path)) {
    run_r_script(
      "gam_cox_nonlinearity.R",
      c(
        "--inputs-dir", INPUTS_DIR,
        "--output-dir", NONLINEAR_OUTPUT_DIR,
        "--landmark-days", as.character(landmark),
        "--feature-selection-csv", selection_path
      )
    )
  } else {
    cat(sprintf("[skip] landmark +%dd nonlinear GAM output exists\n", landmark))
  }
}
stopifnot(all(file.exists(nonlinear_outputs)))


## Nonlinearity summary


In [ ]:
nonlinearity <- do.call(rbind, lapply(nonlinear_outputs, read.csv, check.names = FALSE))
flagged <- nonlinearity[
  !is.na(nonlinearity$q_lrt) & nonlinearity$q_lrt < 0.05 & nonlinearity$edf > 1.5,
  c("landmark_days", "feature", "edf", "p_lrt", "q_lrt", "delta_aic")
]
flagged[order(flagged$q_lrt), ]
